In [ ]:
# ==========================================
# 1. MOUNT DRIVE + DEFINE SHARED PATHS
# ==========================================

import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive"

# Shortcut to retriever folder hosted on tahasecond2.
# Read-only / read-mostly source.
SHARED_RETRIEVER_ROOT = os.path.join(
    DRIVE_ROOT,
    "ct26_qwen3_embedding_8b_lora",
)

# Separate shortcut to reranker output folder hosted on tahasecond2.
# This is where the reranker experiment will write outputs.
SHARED_RERANKER_ROOT = os.path.join(
    DRIVE_ROOT,
    "ct26_qwen3_reranker8b_lora_listwise_top10",
)

assert os.path.isdir(SHARED_RETRIEVER_ROOT), (
    f"Could not find retriever shortcut folder:\n{SHARED_RETRIEVER_ROOT}\n\n"
    "Create/add the shortcut to MyDrive in the Colab Google account."
)

assert os.path.isdir(SHARED_RERANKER_ROOT), (
    f"Could not find reranker shortcut folder:\n{SHARED_RERANKER_ROOT}\n\n"
    "Create this folder in tahasecond2, share it with tahathird3 as Editor, "
    "then add a shortcut to MyDrive in tahathird3."
)

# Existing trained Qwen retriever
BEST_RETRIEVER_DIR = os.path.join(
    SHARED_RETRIEVER_ROOT,
    "best_qwen3_8b_lora_sentence_transformer",
)

RETRIEVER_ARTIFACTS_DIR = os.path.join(
    SHARED_RETRIEVER_ROOT,
    "retrieval_artifacts",
)

assert os.path.isdir(BEST_RETRIEVER_DIR), f"Missing retriever model dir: {BEST_RETRIEVER_DIR}"
assert os.path.isdir(RETRIEVER_ARTIFACTS_DIR), f"Missing retriever artifacts dir: {RETRIEVER_ARTIFACTS_DIR}"

# Stage-2 writes directly into the separate reranker folder.
STAGE2_ROOT = SHARED_RERANKER_ROOT

CHECKPOINT_DIR = os.path.join(STAGE2_ROOT, "trainer_checkpoints")
BEST_RERANKER_DIR = os.path.join(STAGE2_ROOT, "best_qwen3_reranker8b_lora")
CACHE_DIR = os.path.join(STAGE2_ROOT, "cache")
EVAL_DIR = os.path.join(STAGE2_ROOT, "eval")

for p in [STAGE2_ROOT, CHECKPOINT_DIR, BEST_RERANKER_DIR, CACHE_DIR, EVAL_DIR]:
    os.makedirs(p, exist_ok=True)

print("Retriever root:", SHARED_RETRIEVER_ROOT)
print("Reranker root:", SHARED_RERANKER_ROOT)
print("Best retriever dir:", BEST_RETRIEVER_DIR)
print("Retriever artifacts dir:", RETRIEVER_ARTIFACTS_DIR)
print("Best reranker will save to:", BEST_RERANKER_DIR)

Mounted at /content/drive
Retriever root: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora
Reranker root: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10
Best retriever dir: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora/best_qwen3_8b_lora_sentence_transformer
Retriever artifacts dir: /content/drive/MyDrive/ct26_qwen3_embedding_8b_lora/retrieval_artifacts
Best reranker will save to: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/best_qwen3_reranker8b_lora


In [ ]:
# ==========================================
# 2. INSTALL — Qwen3-Reranker-8B LoRA + FlashAttention
# Colab A100 default: Python 3.12 + torch 2.10.0+cu128
# ==========================================

import sys
import subprocess

INSTALLATION_OK = False

def run(cmd):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# Keep Colab's existing torch. Do NOT reinstall torch.
run(f"""{sys.executable} -m pip install -U \
"transformers>=4.51.0" \
"sentence-transformers>=5.0.0" \
"datasets>=2.19.0" \
"accelerate>=0.30.0" \
"peft>=0.12.0" \
safetensors tqdm scikit-learn packaging ninja""")

# Avoid PEFT/torchao compatibility issue on Colab torch 2.10 cu128.
run(f"{sys.executable} -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128")

# Prebuilt FlashAttention wheel for:
# flash-attn 2.8.3 + CUDA 12.8 + torch 2.10 + Python 3.12 + Linux x86_64
FLASH_ATTN_WHEEL = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
run(f'{sys.executable} -m pip install -U "{FLASH_ATTN_WHEEL}"')

# Verification
import torch
import transformers
import flash_attn
from transformers.utils import is_flash_attn_2_available

print("\n===== VERIFICATION =====")
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
print("transformers:", transformers.__version__)
print("flash_attn:", getattr(flash_attn, "__version__", "unknown"))
print("FA2 available to Transformers:", is_flash_attn_2_available())

assert torch.cuda.is_available(), "CUDA is not available"
assert torch.cuda.is_bf16_supported(), "BF16 is not supported"
assert is_flash_attn_2_available(), "FlashAttention 2 is not available to Transformers"

INSTALLATION_OK = True
print("\n✅ INSTALLATION_OK = True")


$ /usr/bin/python3 -m pip install -U "transformers>=4.51.0" "sentence-transformers>=5.0.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.12.0" safetensors tqdm scikit-learn packaging ninja

$ /usr/bin/python3 -m pip install -U torchao --index-url https://download.pytorch.org/whl/cu128

$ /usr/bin/python3 -m pip install -U "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

===== VERIFICATION =====
torch: 2.10.0+cu128
CUDA available: True
CUDA used by torch: 12.8
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True
transformers: 5.8.0
flash_attn: 2.8.3
FA2 available to Transformers: True

✅ INSTALLATION_OK = True


In [ ]:
# ==========================================
# 3. CT26 TASK 1 — QWEN3-RERANKER-8B LORA TRAINING
#
# Design:
# - Candidate depth: top-10 only.
# - Training group: 1 gold + 9 hardest negatives from Qwen retriever top-10.
# - Reranker score: yes_logit - no_logit.
# - Loss: listwise softmax CE over 10 candidates.
# - Saves LoRA adapter + tokenizer + meta into the shared retriever Drive folder.
# ==========================================

assert INSTALLATION_OK is True, "Installation failed. Do not run training."

# ==========================================
# IMPORTS
# ==========================================
import os
import gc
import json
import gzip
import math
import time
import random
import shutil
from datetime import datetime
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from tqdm.auto import tqdm
from torch.utils.data import Dataset as TorchDataset, DataLoader

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
)

from transformers.utils import is_flash_attn_2_available

# ==========================================
# SAFETY CHECKS
# ==========================================
assert torch.cuda.is_available(), "CUDA is required."
assert torch.cuda.is_bf16_supported(), "BF16 is required/recommended for A100."
assert is_flash_attn_2_available(), "FlashAttention 2 is not available."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
print("Stage-2 root:", STAGE2_ROOT)

# ==========================================
# CONFIG
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN", None)

DATASET_NAME = "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims"
LANGUAGES = ["en", "fr", "de"]

RERANKER_MODEL_NAME = "Qwen/Qwen3-Reranker-8B"

# Keep experiment apples-to-apples with current reranker setup.
TOPK = 10
TRAIN_RETRIEVAL_TOPK = TOPK
DEV_RERANK_K = TOPK
LISTWISE_GROUP_SIZE = TOPK
assert LISTWISE_GROUP_SIZE == 10

# Qwen retriever instruction used in your trained retriever.
RETRIEVER_TASK_INSTRUCTION = (
    "Given a scientific web claim, retrieve the title and abstract of the "
    "scientific publication that is the source or best evidence for the claim."
)

# Reranker uses the same task instruction in the Qwen reranker prompt.
RERANKER_TASK_INSTRUCTION = RETRIEVER_TASK_INSTRUCTION

# Practical Qwen3-Reranker-8B settings
RERANKER_MAX_LENGTH = 512

# Training defaults.
# Batch size is in GROUPS. Each group has 10 query-doc pairs.
# For A100 40GB, group batch size 1 is safest.
MAX_EPOCHS = 1
TRAIN_GROUP_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 25

# Save adapter checkpoints during training so an overnight disconnect does not lose everything.
SAVE_EVERY_OPT_STEPS = 0
DELETE_INTERMEDIATE_CHECKPOINTS_AFTER_EXPORT = True

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Candidate/cache behavior
FORCE_REBUILD_TRAIN_CANDIDATES = False
FORCE_REBUILD_DEV_CANDIDATES = False
FORCE_REBUILD_TRAIN_GROUPS = False

# Dev eval behavior
RUN_PRETRAIN_EVAL = False
FUSION_ALPHAS = [round(x, 1) for x in np.linspace(0.0, 1.0, 11)]

# Inference batch autotune for dev scoring
PAIR_EVAL_BATCH_CANDIDATES = [96, 88, 80, 72, 64, 56, 48, 40, 32, 24, 16, 8, 4, 2, 1]

# Paths
TRAIN_CANDIDATES_GZ = os.path.join(CACHE_DIR, f"train_candidates_top{TOPK}.json.gz")
DEV_CANDIDATES_GZ = os.path.join(CACHE_DIR, f"dev_candidates_top{TOPK}.json.gz")
TRAIN_GROUPS_GZ = os.path.join(CACHE_DIR, f"train_listwise_groups_top{TOPK}.json.gz")

TRAINING_HISTORY_PATH = os.path.join(EVAL_DIR, "training_history.json")
FINAL_ALPHA_SUMMARY_PATH = os.path.join(EVAL_DIR, f"best_alpha_sweep_summary_top{TOPK}.json")
FINAL_METRICS_PATH = os.path.join(EVAL_DIR, f"best_qwen3_reranker8b_lora_top{TOPK}_metrics.json")
META_PATH = os.path.join(STAGE2_ROOT, "meta.json")

# Qwen retriever artifact paths
DOC_EMB_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "qwen3_8b_lora_doc_embeddings.npy")
DOC_IDS_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_ids_list.json")
DOC_RAW_PATH = os.path.join(RETRIEVER_ARTIFACTS_DIR, "doc_raw.json.gz")

assert os.path.isfile(DOC_EMB_PATH), f"Missing doc embeddings: {DOC_EMB_PATH}"
assert os.path.isfile(DOC_IDS_PATH), f"Missing doc IDs: {DOC_IDS_PATH}"
assert os.path.isfile(DOC_RAW_PATH), f"Missing doc raw: {DOC_RAW_PATH}"

# ==========================================
# HELPERS
# ==========================================
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def print_gpu_memory(label: str):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_alloc = torch.cuda.max_memory_allocated() / 1024**3
        print(
            f"[GPU MEM] {label}: "
            f"allocated={allocated:.2f}GB reserved={reserved:.2f}GB max_allocated={max_alloc:.2f}GB"
        )

def save_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json_gz(path: str, obj: Any):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json_gz(path: str):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

def normalize_ws(text: Any) -> str:
    return " ".join(str(text).split())

def dataset_kwargs():
    kwargs = {}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    return kwargs

def load_ct26_split(lang: str, split: str):
    return load_dataset(DATASET_NAME, lang, split=split, **dataset_kwargs())

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"

def zscore(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def compute_metrics_from_ranks(ranks):
    ranks = np.asarray(ranks)
    return {
        "MRR@1": float(np.mean([1.0 / r if r <= 1 else 0.0 for r in ranks])),
        "MRR@5": float(np.mean([1.0 / r if r <= 5 else 0.0 for r in ranks])),
        "MRR@10": float(np.mean([1.0 / r if r <= 10 else 0.0 for r in ranks])),
        "Recall@5": float((ranks <= 5).mean()),
        "Recall@10": float((ranks <= 10).mean()),
    }

def print_multilingual_metrics(title, metrics):
    print(f"\n===== {title} =====")
    for lang in LANGUAGES:
        print(f"--- {lang.upper()} ---")
        print(f"  MRR@1     : {metrics[f'{lang}_mrr@1']:.4f}")
        print(f"  MRR@5     : {metrics[f'{lang}_mrr@5']:.4f}")
        print(f"  MRR@10    : {metrics[f'{lang}_mrr@10']:.4f}")
        print(f"  Recall@5  : {metrics[f'{lang}_recall@5']:.4f}")
        print(f"  Recall@10 : {metrics[f'{lang}_recall@10']:.4f}")
    print("===== MULTILINGUAL AVG =====")
    print(f"  MRR@1     : {metrics['multilingual_avg_mrr@1']:.4f}")
    print(f"  MRR@5     : {metrics['multilingual_avg_mrr@5']:.4f}")
    print(f"  MRR@10    : {metrics['multilingual_avg_mrr@10']:.4f}")
    print(f"  Recall@5  : {metrics['multilingual_avg_recall@5']:.4f}")
    print(f"  Recall@10 : {metrics['multilingual_avg_recall@10']:.4f}")

def get_topk_from_scores(scores, k):
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(-part_scores, axis=1)
    top_idx = np.take_along_axis(part, order, axis=1)
    top_scores = np.take_along_axis(part_scores, order, axis=1)
    return top_idx, top_scores

def batched_retrieve_topk(query_texts, retriever, doc_matrix, topk, batch_size=32):
    all_topk_idx = []
    all_topk_scores = []

    for start in tqdm(range(0, len(query_texts), batch_size), desc=f"Qwen retriever top-{topk}"):
        batch_queries = query_texts[start:start + batch_size]

        with torch.inference_mode():
            q_emb = retriever.encode(
                batch_queries,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype(np.float32)

        scores = q_emb @ doc_matrix.T
        batch_topk_idx, batch_topk_scores = get_topk_from_scores(scores, topk)

        all_topk_idx.extend(batch_topk_idx.tolist())
        all_topk_scores.extend(batch_topk_scores.astype(np.float32).tolist())

        del q_emb, scores, batch_topk_idx, batch_topk_scores
        cleanup_cuda()

    return all_topk_idx, all_topk_scores

# ==========================================
# QWEN RERANKER PROMPT + SCORING
# ==========================================
def make_reranker_pair_text(query, doc):
    return f"<Instruct>: {RERANKER_TASK_INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}"

class QwenRerankerPairTokenizer:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.prefix = (
            "<|im_start|>system\n"
            "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
            "Note that the answer can only be \"yes\" or \"no\"."
            "<|im_end|>\n"
            "<|im_start|>user\n"
        )

        self.suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

        self.prefix_tokens = tokenizer.encode(self.prefix, add_special_tokens=False)
        self.suffix_tokens = tokenizer.encode(self.suffix, add_special_tokens=False)

        self.max_pair_tokens = self.max_length - len(self.prefix_tokens) - len(self.suffix_tokens)
        assert self.max_pair_tokens > 32, "RERANKER_MAX_LENGTH is too small."

    def encode_pairs(self, pairs):
        encoded = []

        for q, d in pairs:
            pair_text = make_reranker_pair_text(q, d)
            pair_tokens = self.tokenizer.encode(
                pair_text,
                add_special_tokens=False,
                truncation=True,
                max_length=self.max_pair_tokens,
            )
            input_ids = self.prefix_tokens + pair_tokens + self.suffix_tokens
            encoded.append(input_ids)

        max_len = max(len(x) for x in encoded)
        max_len = int(math.ceil(max_len / 8) * 8)

        pad_id = self.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = self.tokenizer.eos_token_id

        input_ids_batch = []
        attention_mask_batch = []

        for ids in encoded:
            pad_len = max_len - len(ids)

            # Left padding is recommended/safer for decoder-only generation-style scoring.
            input_ids_batch.append([pad_id] * pad_len + ids)
            attention_mask_batch.append([0] * pad_len + [1] * len(ids))

        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.long),
        }

def forward_yes_no_scores(model, batch, token_yes_id, token_no_id):
    # num_logits_to_keep=1 avoids returning logits for every sequence position.
    try:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            num_logits_to_keep=1,
        )
    except TypeError:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

    logits = outputs.logits[:, -1, :]
    yes_logits = logits[:, token_yes_id]
    no_logits = logits[:, token_no_id]
    return yes_logits - no_logits

def estimate_pair_len(pair):
    q, d = pair
    return len(q.split()) + len(d.split())

def score_flat_pairs_qwen_sorted(
    model,
    pair_tokenizer,
    pairs,
    batch_size,
    device,
    token_yes_id,
    token_no_id,
    desc,
):
    n = len(pairs)
    if n == 0:
        return np.zeros((0,), dtype=np.float32)

    was_training = model.training
    model.eval()

    try:
        lengths = np.asarray([estimate_pair_len(p) for p in pairs], dtype=np.int32)
        order = np.argsort(lengths)
        sorted_pairs = [pairs[i] for i in order]

        sorted_scores = np.empty(n, dtype=np.float32)

        with torch.inference_mode():
            for start in tqdm(range(0, n, batch_size), desc=desc):
                batch_pairs = sorted_pairs[start:start + batch_size]
                batch = pair_tokenizer.encode_pairs(batch_pairs)
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
                    batch_scores = forward_yes_no_scores(
                        model=model,
                        batch=batch,
                        token_yes_id=token_yes_id,
                        token_no_id=token_no_id,
                    )

                vals = batch_scores.detach().float().cpu().numpy().astype(np.float32)
                sorted_scores[start:start + len(vals)] = vals

                del batch, batch_scores, vals
                cleanup_cuda()

        unsorted_scores = np.empty(n, dtype=np.float32)
        unsorted_scores[order] = sorted_scores
        return unsorted_scores

    finally:
        if was_training:
            model.train()

def autotune_pair_batch_size(model, pair_tokenizer, sample_pairs, device, token_yes_id, token_no_id):
    print("\nAutotuning dev inference pair batch size...")
    model.eval()

    # Sort longest first for safer tuning.
    sample_pairs = sorted(sample_pairs, key=estimate_pair_len, reverse=True)

    for bs in PAIR_EVAL_BATCH_CANDIDATES:
        try:
            cleanup_cuda()
            probe = sample_pairs[:bs]
            batch = pair_tokenizer.encode_pairs(probe)
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
                    out = forward_yes_no_scores(model, batch, token_yes_id, token_no_id)

            _ = out.detach().float().cpu().numpy()
            del batch, out
            cleanup_cuda()

            print(f"Autotuned dev PAIR_EVAL_BATCH_SIZE = {bs}")
            return bs

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"Batch size {bs} OOM. Trying smaller...")
                cleanup_cuda()
            else:
                raise

    raise RuntimeError("Could not find safe inference batch size.")

# ==========================================
# LOAD RETRIEVER ARTIFACTS
# ==========================================
print("\nLoading Qwen retriever artifacts...")
doc_matrix = np.load(DOC_EMB_PATH).astype(np.float32)
doc_ids_list = load_json(DOC_IDS_PATH)
doc_raw = load_json_gz(DOC_RAW_PATH)

doc_ids_list = [str(x) for x in doc_ids_list]
doc_raw = {str(k): v for k, v in doc_raw.items()}
docid_to_idx = {did: idx for idx, did in enumerate(doc_ids_list)}

print("Doc matrix shape:", doc_matrix.shape)
print("Documents:", len(doc_ids_list))

# ==========================================
# LOAD TRAIN/DEV SPLITS
# ==========================================
print("\nLoading train/dev splits...")
combined_train = []
dev_by_lang_raw = {}

for lang in LANGUAGES:
    train_split = load_ct26_split(lang, "train")
    dev_split = load_ct26_split(lang, "dev")

    for item in train_split:
        row = dict(item)
        row["lang"] = lang
        combined_train.append(row)

    dev_by_lang_raw[lang] = [dict(x) for x in dev_split]

    print(f"{lang}: train={len(train_split)}, dev={len(dev_split)}")

print("Total train rows:", len(combined_train))

# ==========================================
# BUILD TRAIN/DEV CANDIDATES WITH QWEN RETRIEVER
# ==========================================
def build_candidates_for_rows(rows, retriever, doc_matrix, doc_ids_list, doc_raw, topk, batch_size, split_name):
    queries_raw = [normalize_ws(x["text"]) for x in rows]
    queries_for_retriever = [
        get_detailed_instruct(RETRIEVER_TASK_INSTRUCTION, q)
        for q in queries_raw
    ]
    gold_ids = [str(x["pubkey"]) for x in rows]

    topk_idx, topk_scores = batched_retrieve_topk(
        query_texts=queries_for_retriever,
        retriever=retriever,
        doc_matrix=doc_matrix,
        topk=topk,
        batch_size=batch_size,
    )

    samples = []

    for i, gold_id in enumerate(tqdm(gold_ids, desc=f"Build {split_name} samples")):
        candidate_ids = [doc_ids_list[idx] for idx in topk_idx[i]]
        candidate_texts = [doc_raw[did] for did in candidate_ids]

        samples.append({
            "query": queries_raw[i],
            "query_with_retriever_instruction": queries_for_retriever[i],
            "gold_id": gold_id,
            "candidate_ids": candidate_ids,
            "candidate_texts": candidate_texts,
            "dense_scores": topk_scores[i],
            "retriever_scores": topk_scores[i],
            "lang": rows[i].get("lang", None),
        })

    return samples

need_train_candidates = FORCE_REBUILD_TRAIN_CANDIDATES or not os.path.isfile(TRAIN_CANDIDATES_GZ)
need_dev_candidates = FORCE_REBUILD_DEV_CANDIDATES or not os.path.isfile(DEV_CANDIDATES_GZ)

if need_train_candidates or need_dev_candidates:
    print("\nLoading Qwen LoRA retriever for candidate generation...")
    print_gpu_memory("before retriever load")

    try:
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs={
                "attn_implementation": "flash_attention_2",
                "dtype": torch.bfloat16,
                "device_map": {"": 0},
            },
            processor_kwargs={
                "padding_side": "left",
            },
        )
    except TypeError:
        retriever = SentenceTransformer(
            BEST_RETRIEVER_DIR,
            model_kwargs={
                "attn_implementation": "flash_attention_2",
                "torch_dtype": torch.bfloat16,
                "device_map": {"": 0},
            },
            tokenizer_kwargs={
                "padding_side": "left",
            },
        )

    retriever.max_seq_length = 512
    retriever.tokenizer.padding_side = "left"
    retriever.eval()

    print_gpu_memory("after retriever load")

    if need_train_candidates:
        print("\nBuilding train top-10 candidates...")
        train_candidates = build_candidates_for_rows(
            rows=combined_train,
            retriever=retriever,
            doc_matrix=doc_matrix,
            doc_ids_list=doc_ids_list,
            doc_raw=doc_raw,
            topk=TRAIN_RETRIEVAL_TOPK,
            batch_size=32,
            split_name="train",
        )
        save_json_gz(TRAIN_CANDIDATES_GZ, train_candidates)
        print("Saved train candidates:", TRAIN_CANDIDATES_GZ)
    else:
        print("Using cached train candidates:", TRAIN_CANDIDATES_GZ)

    if need_dev_candidates:
        print("\nBuilding dev top-10 candidates...")
        dev_candidates_by_lang = {}

        for lang in LANGUAGES:
            dev_candidates_by_lang[lang] = build_candidates_for_rows(
                rows=dev_by_lang_raw[lang],
                retriever=retriever,
                doc_matrix=doc_matrix,
                doc_ids_list=doc_ids_list,
                doc_raw=doc_raw,
                topk=DEV_RERANK_K,
                batch_size=32,
                split_name=f"dev-{lang}",
            )

        save_json_gz(DEV_CANDIDATES_GZ, dev_candidates_by_lang)
        print("Saved dev candidates:", DEV_CANDIDATES_GZ)
    else:
        print("Using cached dev candidates:", DEV_CANDIDATES_GZ)

    print("\nUnloading retriever before loading reranker...")
    try:
        retriever.cpu()
    except Exception:
        pass

    del retriever
    cleanup_cuda()
    print_gpu_memory("after retriever unload")

# Load cached candidates
train_candidates = load_json_gz(TRAIN_CANDIDATES_GZ)
dev_candidates_by_lang = load_json_gz(DEV_CANDIDATES_GZ)

print("Train candidates:", len(train_candidates))
for lang in LANGUAGES:
    print(f"Dev candidates {lang}:", len(dev_candidates_by_lang[lang]))

# ==========================================
# BUILD LISTWISE TRAIN GROUPS: 1 GOLD + 9 HARD NEGATIVES
# ==========================================
def build_train_listwise_groups_from_candidates(samples, doc_raw, group_size=10, seed=42):
    groups = []
    skipped = 0
    injected_gold = 0
    gold_already_present = 0

    for i, sample in enumerate(tqdm(samples, desc="Building listwise train groups")):
        q = normalize_ws(sample["query"])
        gold_id = str(sample["gold_id"])
        candidate_ids = [str(x) for x in sample["candidate_ids"]]

        if gold_id not in doc_raw:
            skipped += 1
            continue

        wrong_ids = [did for did in candidate_ids if did != gold_id]

        if len(wrong_ids) < group_size - 1:
            skipped += 1
            continue

        if gold_id in candidate_ids:
            # Use the retrieved top-10 set if gold is already there.
            group_ids = candidate_ids[:group_size]
            if gold_id not in group_ids:
                # Edge case: gold found below group_size, inject it.
                group_ids = [gold_id] + wrong_ids[:group_size - 1]
                injected_gold += 1
            else:
                gold_already_present += 1
        else:
            # Gold was missed by retriever top-10. Inject gold + top 9 retrieved hard negatives.
            group_ids = [gold_id] + wrong_ids[:group_size - 1]
            injected_gold += 1

        # Guarantee exactly group_size and exactly one gold.
        group_ids = group_ids[:group_size]
        if gold_id not in group_ids:
            group_ids[-1] = gold_id

        # Deterministic shuffle to avoid position bias.
        rng = np.random.default_rng(seed + i)
        perm = rng.permutation(group_size)
        shuffled_ids = [group_ids[j] for j in perm]
        positive_index = shuffled_ids.index(gold_id)

        groups.append({
            "query": q,
            "candidate_ids": shuffled_ids,
            "candidate_texts": [doc_raw[did] for did in shuffled_ids],
            "positive_index": positive_index,
            "gold_id": gold_id,
            "lang": sample.get("lang", None),
        })

    print("Train groups:", len(groups))
    print("Skipped:", skipped)
    print("Gold already in retrieved group:", gold_already_present)
    print("Gold injected:", injected_gold)

    return groups

if FORCE_REBUILD_TRAIN_GROUPS or not os.path.isfile(TRAIN_GROUPS_GZ):
    train_groups = build_train_listwise_groups_from_candidates(
        samples=train_candidates,
        doc_raw=doc_raw,
        group_size=LISTWISE_GROUP_SIZE,
        seed=SEED,
    )
    save_json_gz(TRAIN_GROUPS_GZ, train_groups)
    print("Saved train groups:", TRAIN_GROUPS_GZ)
else:
    train_groups = load_json_gz(TRAIN_GROUPS_GZ)
    print("Loaded train groups:", len(train_groups))

assert len(train_groups) > 0, "No train groups were built."

# ==========================================
# DATASET + COLLATOR
# ==========================================
class ListwiseQwenDataset(TorchDataset):
    def __init__(self, groups):
        self.groups = groups

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        return self.groups[idx]

class ListwiseQwenCollator:
    def __init__(self, pair_tokenizer, group_size):
        self.pair_tokenizer = pair_tokenizer
        self.group_size = group_size

    def __call__(self, batch):
        pairs = []
        positive_indices = []

        for item in batch:
            assert len(item["candidate_texts"]) == self.group_size
            q = item["query"]
            for d in item["candidate_texts"]:
                pairs.append((q, d))
            positive_indices.append(item["positive_index"])

        features = self.pair_tokenizer.encode_pairs(pairs)
        features["positive_indices"] = torch.tensor(positive_indices, dtype=torch.long)
        features["num_groups"] = len(batch)
        return features

# ==========================================
# LOAD QWEN3-RERANKER-8B + LORA
# ==========================================
print("\nLoading Qwen3-Reranker-8B...")
print_gpu_memory("before reranker load")

tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_NAME,
    padding_side="left",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

token_yes_id = tokenizer.encode("yes", add_special_tokens=False)[0]
token_no_id = tokenizer.encode("no", add_special_tokens=False)[0]

print("token_yes_id:", token_yes_id, "| token:", tokenizer.decode([token_yes_id]))
print("token_no_id:", token_no_id, "| token:", tokenizer.decode([token_no_id]))

def load_qwen_causal_lm_bf16(model_name):
    try:
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.bfloat16,
            attn_implementation="flash_attention_2",
            device_map={"": 0},
            trust_remote_code=True,
        )
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            attn_implementation="flash_attention_2",
            device_map={"": 0},
            trust_remote_code=True,
        )

model = load_qwen_causal_lm_bf16(RERANKER_MODEL_NAME)

model.config.use_cache = False

try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    print("Enabled gradient checkpointing with use_reentrant=False.")
except TypeError:
    model.gradient_checkpointing_enable()
    print("Enabled gradient checkpointing.")
except Exception as e:
    print("Could not enable gradient checkpointing:", repr(e))

try:
    model.enable_input_require_grads()
    print("Enabled input grads for PEFT + gradient checkpointing.")
except Exception as e:
    print("Could not enable input grads:", repr(e))

print("\nAdding LoRA adapter...")
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

device = torch.device("cuda")
model.train()

pair_tokenizer = QwenRerankerPairTokenizer(
    tokenizer=tokenizer,
    max_length=RERANKER_MAX_LENGTH,
)

print_gpu_memory("after reranker + LoRA load")

# ==========================================
# DATALOADER
# ==========================================
train_dataset = ListwiseQwenDataset(train_groups)
train_collator = ListwiseQwenCollator(
    pair_tokenizer=pair_tokenizer,
    group_size=LISTWISE_GROUP_SIZE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_GROUP_BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collator,
    num_workers=0,
    pin_memory=False,
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
max_train_steps = num_update_steps_per_epoch * MAX_EPOCHS
warmup_steps = int(max_train_steps * WARMUP_RATIO)

print("\nTraining schedule:")
print("Train groups:", len(train_dataset))
print("Train group batch size:", TRAIN_GROUP_BATCH_SIZE)
print("Pairs per forward step:", TRAIN_GROUP_BATCH_SIZE * LISTWISE_GROUP_SIZE)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("Optimizer steps per epoch:", num_update_steps_per_epoch)
print("Max epochs:", MAX_EPOCHS)
print("Max optimizer steps:", max_train_steps)
print("Warmup steps:", warmup_steps)

# ==========================================
# OPTIMIZER + SCHEDULER
# ==========================================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=max_train_steps,
)

# ==========================================
# EVAL FUNCTIONS
# ==========================================
def evaluate_scored_samples(scored_samples_by_lang, alpha=None):
    metrics = {}
    per_lang = {}

    for lang in LANGUAGES:
        samples = scored_samples_by_lang[lang]
        ranks = []

        for sample in samples:
            gold_id = sample["gold_id"]
            candidate_ids = [str(x) for x in sample["candidate_ids"]]
            ce_scores = np.asarray(sample["ce_scores"], dtype=np.float32)

            if alpha is None:
                final_scores = ce_scores
            else:
                dense_scores = np.asarray(sample["dense_scores"], dtype=np.float32)
                final_scores = alpha * zscore(dense_scores) + (1.0 - alpha) * zscore(ce_scores)

            order = np.argsort(-final_scores)
            reranked_ids = [candidate_ids[i] for i in order]
            rank = reranked_ids.index(gold_id) + 1 if gold_id in reranked_ids else 10000
            ranks.append(rank)

        lang_scores = compute_metrics_from_ranks(ranks)
        per_lang[lang] = lang_scores

        metrics[f"{lang}_mrr@1"] = lang_scores["MRR@1"]
        metrics[f"{lang}_mrr@5"] = lang_scores["MRR@5"]
        metrics[f"{lang}_mrr@10"] = lang_scores["MRR@10"]
        metrics[f"{lang}_recall@5"] = lang_scores["Recall@5"]
        metrics[f"{lang}_recall@10"] = lang_scores["Recall@10"]

    metrics["multilingual_avg_mrr@1"] = float(np.mean([per_lang[l]["MRR@1"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@5"] = float(np.mean([per_lang[l]["MRR@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_mrr@10"] = float(np.mean([per_lang[l]["MRR@10"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@5"] = float(np.mean([per_lang[l]["Recall@5"] for l in LANGUAGES]))
    metrics["multilingual_avg_recall@10"] = float(np.mean([per_lang[l]["Recall@10"] for l in LANGUAGES]))

    return metrics

def score_dev_candidates(model, epoch_label, pair_batch_size):
    scored = {}

    print(f"\nScoring dev candidates with reranker | {epoch_label}")

    for lang in LANGUAGES:
        samples = dev_candidates_by_lang[lang]

        flat_pairs = []
        counts = []

        for sample in samples:
            q = sample["query"]
            docs = sample["candidate_texts"]
            counts.append(len(docs))
            flat_pairs.extend((q, d) for d in docs)

        flat_scores = score_flat_pairs_qwen_sorted(
            model=model,
            pair_tokenizer=pair_tokenizer,
            pairs=flat_pairs,
            batch_size=pair_batch_size,
            device=device,
            token_yes_id=token_yes_id,
            token_no_id=token_no_id,
            desc=f"Score dev {lang.upper()} | {epoch_label}",
        )

        scored_samples = []
        offset = 0

        for sample, count in zip(samples, counts):
            new_sample = dict(sample)
            new_sample["ce_scores"] = flat_scores[offset:offset + count].tolist()
            scored_samples.append(new_sample)
            offset += count

        scored[lang] = scored_samples

    return scored

def run_alpha_sweep(scored_samples_by_lang, title, save_prefix):
    print(f"\n===== ALPHA SWEEP | {title} =====")

    pure_metrics = evaluate_scored_samples(scored_samples_by_lang, alpha=None)
    print_multilingual_metrics(f"{title} | pure reranker", pure_metrics)

    alpha_summary = []
    alpha_results = {}

    for alpha in FUSION_ALPHAS:
        metrics = evaluate_scored_samples(scored_samples_by_lang, alpha=alpha)
        key = f"{alpha:.1f}"
        alpha_results[key] = metrics

        row = {
            "alpha": alpha,
            "multilingual_avg_mrr@1": metrics["multilingual_avg_mrr@1"],
            "multilingual_avg_mrr@5": metrics["multilingual_avg_mrr@5"],
            "multilingual_avg_mrr@10": metrics["multilingual_avg_mrr@10"],
            "multilingual_avg_recall@5": metrics["multilingual_avg_recall@5"],
            "multilingual_avg_recall@10": metrics["multilingual_avg_recall@10"],
        }
        alpha_summary.append(row)

        print(
            f"alpha={alpha:.1f} | "
            f"MRR@5={metrics['multilingual_avg_mrr@5']:.4f} | "
            f"MRR@10={metrics['multilingual_avg_mrr@10']:.4f} | "
            f"Recall@5={metrics['multilingual_avg_recall@5']:.4f} | "
            f"Recall@10={metrics['multilingual_avg_recall@10']:.4f}"
        )

    best_by_mrr5 = max(alpha_summary, key=lambda x: x["multilingual_avg_mrr@5"])

    print(f"\n===== BEST ALPHA | {title} =====")
    for k, v in best_by_mrr5.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    out = {
        "title": title,
        "pure_reranker_metrics": pure_metrics,
        "alpha_results": alpha_results,
        "alpha_summary": alpha_summary,
        "best_by_mrr5": best_by_mrr5,
    }

    save_json(os.path.join(EVAL_DIR, f"{save_prefix}_alpha_sweep.json"), out)
    return out

def save_adapter_checkpoint(path, model, tokenizer, extra_meta=None):
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

    if extra_meta is not None:
        save_json(os.path.join(path, "ct26_checkpoint_meta.json"), extra_meta)

    print("Saved adapter checkpoint:", path)

# Autotune dev inference batch size once.
probe_pairs = []
for lang in LANGUAGES:
    for sample in dev_candidates_by_lang[lang][:8]:
        q = sample["query"]
        for d in sample["candidate_texts"]:
            probe_pairs.append((q, d))
    if len(probe_pairs) >= 80:
        break

DEV_PAIR_BATCH_SIZE = autotune_pair_batch_size(
    model=model,
    pair_tokenizer=pair_tokenizer,
    sample_pairs=probe_pairs,
    device=device,
    token_yes_id=token_yes_id,
    token_no_id=token_no_id,
)

if RUN_PRETRAIN_EVAL:
    pre_scored = score_dev_candidates(model, "pretrain", DEV_PAIR_BATCH_SIZE)
    save_json_gz(os.path.join(EVAL_DIR, f"dev_scored_pretrain_top{TOPK}.json.gz"), pre_scored)
    pre_eval = run_alpha_sweep(pre_scored, "pretrain", "pretrain")

# ==========================================
# TRAIN LOOP
# ==========================================
print("\nStarting Qwen3-Reranker-8B LoRA listwise training...")

best_metric = -1.0
best_epoch = None
best_alpha = None
best_eval_payload = None
history = []

global_update_step = 0
raw_forward_step = 0
optimizer.zero_grad(set_to_none=True)

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_loss_count = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}")

    for batch_idx, batch in enumerate(progress, start=1):
        positive_indices = batch.pop("positive_indices").to(device)
        num_groups = int(batch.pop("num_groups"))

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
            scores = forward_yes_no_scores(
                model=model,
                batch=batch,
                token_yes_id=token_yes_id,
                token_no_id=token_no_id,
            )

            logits = scores.view(num_groups, LISTWISE_GROUP_SIZE)
            loss = F.cross_entropy(logits.float(), positive_indices)

        loss_value = float(loss.detach().cpu().item())
        epoch_loss_sum += loss_value
        epoch_loss_count += 1

        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        raw_forward_step += 1

        should_step = (batch_idx % GRAD_ACCUM_STEPS == 0) or (batch_idx == len(train_loader))

        if should_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_update_step += 1

            if SAVE_EVERY_OPT_STEPS and global_update_step % SAVE_EVERY_OPT_STEPS == 0:
                ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-step{global_update_step}")
                save_adapter_checkpoint(
                    ckpt_dir,
                    model,
                    tokenizer,
                    extra_meta={
                        "epoch": epoch,
                        "global_update_step": global_update_step,
                        "raw_forward_step": raw_forward_step,
                        "loss_recent": loss_value,
                        "base_model": RERANKER_MODEL_NAME,
                        "stage2_root": STAGE2_ROOT,
                    },
                )

        if batch_idx % LOGGING_STEPS == 0 or batch_idx == len(train_loader):
            avg_loss = epoch_loss_sum / max(epoch_loss_count, 1)
            lr_now = scheduler.get_last_lr()[0]
            progress.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{lr_now:.2e}",
                opt_step=global_update_step,
            )

        del batch, positive_indices, scores, logits, loss
        if batch_idx % 200 == 0:
            gc.collect()

    train_loss_avg = epoch_loss_sum / max(epoch_loss_count, 1)
    print(f"\nEpoch {epoch} train loss: {train_loss_avg:.6f}")

    # Save epoch checkpoint before evaluation.
    epoch_ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-epoch{epoch}")
    save_adapter_checkpoint(
        epoch_ckpt_dir,
        model,
        tokenizer,
        extra_meta={
            "epoch": epoch,
            "global_update_step": global_update_step,
            "train_loss": train_loss_avg,
            "base_model": RERANKER_MODEL_NAME,
            "stage2_root": STAGE2_ROOT,
        },
    )

    # Dev eval
    scored_epoch = score_dev_candidates(
        model=model,
        epoch_label=f"epoch{epoch}",
        pair_batch_size=DEV_PAIR_BATCH_SIZE,
    )

    scored_epoch_path = os.path.join(EVAL_DIR, f"dev_scored_epoch{epoch}_top{TOPK}.json.gz")
    save_json_gz(scored_epoch_path, scored_epoch)
    print("Saved scored dev:", scored_epoch_path)

    eval_payload = run_alpha_sweep(
        scored_samples_by_lang=scored_epoch,
        title=f"epoch{epoch}",
        save_prefix=f"epoch{epoch}",
    )

    current_metric = eval_payload["best_by_mrr5"]["multilingual_avg_mrr@5"]
    current_alpha = eval_payload["best_by_mrr5"]["alpha"]

    history_row = {
        "epoch": epoch,
        "train_loss": train_loss_avg,
        "global_update_step": global_update_step,
        "scored_dev_path": scored_epoch_path,
        "best_alpha": current_alpha,
        "best_multilingual_avg_mrr@5": current_metric,
        "best_multilingual_avg_mrr@10": eval_payload["best_by_mrr5"]["multilingual_avg_mrr@10"],
        "best_multilingual_avg_recall@5": eval_payload["best_by_mrr5"]["multilingual_avg_recall@5"],
        "best_multilingual_avg_recall@10": eval_payload["best_by_mrr5"]["multilingual_avg_recall@10"],
    }
    history.append(history_row)
    save_json(TRAINING_HISTORY_PATH, history)

    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        best_alpha = current_alpha
        best_eval_payload = eval_payload

        if os.path.isdir(BEST_RERANKER_DIR):
            shutil.rmtree(BEST_RERANKER_DIR)

        save_adapter_checkpoint(
            BEST_RERANKER_DIR,
            model,
            tokenizer,
            extra_meta={
                "best_epoch": best_epoch,
                "best_alpha": best_alpha,
                "best_multilingual_avg_mrr@5": best_metric,
                "base_model": RERANKER_MODEL_NAME,
                "loss_type": "listwise_softmax_ce_yes_minus_no",
                "topk": TOPK,
                "stage2_root": STAGE2_ROOT,
                "retriever_root": SHARED_RETRIEVER_ROOT,
            },
        )

        save_json(FINAL_METRICS_PATH, best_eval_payload)
        save_json(FINAL_ALPHA_SUMMARY_PATH, best_eval_payload["alpha_summary"])

        print(
            f"\nNew best model saved | epoch={best_epoch} | "
            f"alpha={best_alpha:.1f} | MRR@5={best_metric:.4f}"
        )

elapsed_hours = (time.time() - start_time) / 3600.0

print("\nTraining complete.")
print("Elapsed hours:", round(elapsed_hours, 2))
print("Best epoch:", best_epoch)
print("Best alpha:", best_alpha)
print("Best multilingual avg MRR@5:", best_metric)
print("Best reranker adapter dir:", BEST_RERANKER_DIR)

if DELETE_INTERMEDIATE_CHECKPOINTS_AFTER_EXPORT:
    print("\nDeleting intermediate checkpoints...")
    for name in os.listdir(CHECKPOINT_DIR):
        path = os.path.join(CHECKPOINT_DIR, name)
        if os.path.isdir(path) and name.startswith("checkpoint-"):
            shutil.rmtree(path)

# ==========================================
# SAVE FINAL META
# ==========================================
meta = {
    "dataset_name": DATASET_NAME,
    "languages": LANGUAGES,

    "stage1_retriever": {
        "shared_retriever_root": SHARED_RETRIEVER_ROOT,
        "best_retriever_dir": BEST_RETRIEVER_DIR,
        "retriever_artifacts_dir": RETRIEVER_ARTIFACTS_DIR,
        "doc_embeddings": DOC_EMB_PATH,
        "doc_ids": DOC_IDS_PATH,
        "doc_raw": DOC_RAW_PATH,
        "retriever_task_instruction": RETRIEVER_TASK_INSTRUCTION,
    },

    "stage2_reranker": {
        "shared_reranker_root": SHARED_RERANKER_ROOT,
        "stage2_root": STAGE2_ROOT,
        "base_reranker_model": RERANKER_MODEL_NAME,
        "best_reranker_dir": BEST_RERANKER_DIR,
        "checkpoint_dir": CHECKPOINT_DIR,
        "cache_dir": CACHE_DIR,
        "eval_dir": EVAL_DIR,
        "reranker_task_instruction": RERANKER_TASK_INSTRUCTION,
        "score_type": "yes_logit_minus_no_logit",
        "training_loss": "listwise_softmax_cross_entropy",
        "topk": TOPK,
        "listwise_group_size": LISTWISE_GROUP_SIZE,
        "reranker_max_length": RERANKER_MAX_LENGTH,
    },

    "training": {
        "max_epochs": MAX_EPOCHS,
        "train_group_batch_size": TRAIN_GROUP_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "effective_groups_per_update": TRAIN_GROUP_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "pairs_per_forward": TRAIN_GROUP_BATCH_SIZE * LISTWISE_GROUP_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "max_train_steps": max_train_steps,
        "max_grad_norm": MAX_GRAD_NORM,
        "bf16": True,
        "flash_attention_2": True,
    },

    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "task_type": "CAUSAL_LM",
        "saved_format": "PEFT LoRA adapter, not full 8B base weights",
    },

    "files": {
        "train_candidates": TRAIN_CANDIDATES_GZ,
        "dev_candidates": DEV_CANDIDATES_GZ,
        "train_groups": TRAIN_GROUPS_GZ,
        "training_history": TRAINING_HISTORY_PATH,
        "final_metrics": FINAL_METRICS_PATH,
        "final_alpha_summary": FINAL_ALPHA_SUMMARY_PATH,
        "meta": META_PATH,
    },

    "best_result": {
        "best_epoch": best_epoch,
        "best_alpha": best_alpha,
        "best_multilingual_avg_mrr@5": best_metric,
    },

    "notes_for_test_time": {
        "load_base_model": RERANKER_MODEL_NAME,
        "then_load_adapter_from": BEST_RERANKER_DIR,
        "retriever_model_dir": BEST_RETRIEVER_DIR,
        "retriever_doc_embeddings": DOC_EMB_PATH,
        "candidate_depth": TOPK,
        "fusion_alpha_to_use": best_alpha,
    },
}

save_json(META_PATH, meta)

print("\nSaved final files:")
print("Best reranker adapter:", BEST_RERANKER_DIR)
print("Train candidates:", TRAIN_CANDIDATES_GZ)
print("Dev candidates:", DEV_CANDIDATES_GZ)
print("Train groups:", TRAIN_GROUPS_GZ)
print("Training history:", TRAINING_HISTORY_PATH)
print("Final metrics:", FINAL_METRICS_PATH)
print("Meta:", META_PATH)

cleanup_cuda()
print("\nDone.")

torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
BF16: True
Stage-2 root: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10

Loading Qwen retriever artifacts...
Doc matrix shape: (10000, 4096)
Documents: 10000

Loading train/dev splits...


README.md: 0.00B [00:00, ?B/s]

en_train.json: 0.00B [00:00, ?B/s]

en_dev.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14977 [00:00<?, ? examples/s]

...


Epoch 1 train loss: 0.687593
Saved adapter checkpoint: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/trainer_checkpoints/checkpoint-epoch1

Scoring dev candidates with reranker | epoch1


Score dev EN | epoch1:   0%|          | 0/407 [00:00<?, ?it/s]

Score dev FR | epoch1:   0%|          | 0/74 [00:00<?, ?it/s]

Score dev DE | epoch1:   0%|          | 0/41 [00:00<?, ?it/s]

Saved scored dev: /content/drive/MyDrive/ct26_qwen3_reranker8b_lora_listwise_top10/eval/dev_scored_epoch1_top10.json.gz

===== ALPHA SWEEP | epoch1 =====

===== epoch1 | pure reranker =====
--- EN ---
  MRR@1     : 0.7001
  MRR@5     : 0.7484
  MRR@10    : 0.7522
  Recall@5  : 0.8230
  Recall@10 : 0.8510
--- FR ---
  MRR@1     : 0.7393
  MRR@5     : 0.7861
  MRR@10    : 0.7880
  Recall@5  : 0.8504
  Recall@10 : 0.8632
--- DE ---
  MRR@1     : 0.6295
  MRR@5     : 0.6855
  MRR@10    : 0.6893
  Recall@5  : 0.7642
  Recall@10 : 0.7902
===== MULTILINGUAL AVG =====
  MRR@1     : 0.6897
  MRR@5     : 0.7400
  MRR@10    : 0.7432
  Recall@5  : 0.8126
  Recall@10 : 0.8348
alpha=0.0 | MRR@5=0.7400 | MRR@10=0.7432 | Recall@5=0.8126 | Recall@10=0.8348
alpha=0.1 | MRR@5=0.7425 | MRR@10=0.7456 | Recall@5=0.8127 | Recall@10=0.8348
alpha=0.2 | MRR@5=0.7435 | MRR@10=0.7466 | Recall@5=0.8132 | Recall@10=0.8348
alpha=0.3 | MRR@5=0.7408 | MRR@10=0.7441 | Recall@5=0.8123 | Recall@10=0.8348
alpha=0.4 | MRR@

In [ ]:
from google.colab import runtime
runtime.unassign()